# E11 House Prices：高维混合、缺失含义与集成

> Issue #9 / Epic #6 / 路线图 E11

## 案例背景
Kaggle House Prices 是典型的**表格回归**：数值 + 高基数类别混合，缺失常带业务语义（如「无车库」时 Garage 列同时缺失）。  
竞赛常用 **log1p(SalePrice)** 训练以缓解右偏，提交时再 `expm1` 还原。

## 你将学到
1. 缺失指示与**高基数类别**的频率 / 目标编码思路（train-only）
2. **log 目标**与 RMSE 在价格空间解读
3. **wanglaoshi `ModelEnsemble.blending`**：树模型 + 线性模型互补
4. `SubmissionGenerator`：`prediction_type='label'`（价格点预测，非概率）

## 理论：为何 log 目标
房价右偏：绝对误差对大户型权重过高。对 \(\log(1+y)\) 优化近似相对误差，更稳定。  
集成动机：线性模型抓全局趋势，树模型抓非线性与交互。

## 数据
- 本地：`data/house_prices_train.csv`
- 回退：OpenML / 合成 DEMO


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/house_prices_train.csv')
DEMO = False
if path.exists():
    train = pd.read_csv(path)
    print('local', train.shape)
else:
    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml('house_prices', version=1, as_frame=True, parser='auto')
        train = raw.frame.copy()
        print('OpenML', train.shape)
    except Exception:
        DEMO = True
        print('合成 DEMO 房价数据')
        rng = np.random.RandomState(RANDOM_STATE)
        n = 800
        neighborhoods = [f'Nb_{i}' for i in range(25)]
        train = pd.DataFrame({
            'Id': np.arange(1, n + 1),
            'LotArea': rng.randint(1000, 20000, n),
            'OverallQual': rng.randint(1, 11, n),
            'YearBuilt': rng.randint(1900, 2011, n),
            'GarageArea': rng.choice([0, 200, 400, 600], n, p=[0.15, 0.3, 0.3, 0.25]),
            'Neighborhood': rng.choice(neighborhoods, n),
            'SalePrice': None,
        })
        train['SalePrice'] = (
            5e4 + 30 * train['LotArea'] + 2e4 * train['OverallQual']
            + 50 * train['GarageArea'] + rng.normal(0, 2e4, n)
        ).clip(3e4, None)

if 'SalePrice' not in train.columns:
    for c in train.columns:
        if 'sale' in c.lower() or c.lower() == 'price':
            train = train.rename(columns={c: 'SalePrice'})
            break
if QUICK and len(train) > 1200:
    train = train.sample(1200, random_state=RANDOM_STATE).reset_index(drop=True)
print('DEMO', DEMO)
train.head(3)


## 2. EDA：缺失可能有含义


In [ ]:
from wanglaoshi import DataAnalyzer

eda_df = train.sample(min(len(train), 3000), random_state=RANDOM_STATE) if QUICK else train
analyzer = DataAnalyzer(eda_df)
miss = analyzer.missing_value_analysis()
display(miss.head(20))
garage_cols = [c for c in train.columns if c.startswith('Garage')]
print('garage-related cols (sample)', garage_cols[:8])


## 3. log1p 目标与划分


In [ ]:
from sklearn.model_selection import train_test_split

work = train.copy()
y = np.log1p(work['SalePrice'].astype(float))
X = work.drop(columns=['SalePrice'], errors='ignore')
if 'Id' in X.columns:
    X = X.drop(columns=['Id'])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('log-target mean', ytr.mean())


## 4. 预处理：数值中位数 + 高基数类别频率编码

高基数 `Neighborhood` 等不宜直接 One-Hot。频率编码用**训练集**类别占比映射（未见类别 → 0）。  
也可用 `FE.frequency_encode`；统计量同样必须 train-only。


In [ ]:
from wanglaoshi import FeatureEngineering as FE

num_cols = Xtr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in Xtr.columns if c not in num_cols]
# 取缺失率适中且高基数的若干类别列
card = {c: Xtr[c].astype(str).nunique() for c in cat_cols}
cat_use = sorted(card, key=card.get, reverse=True)[:12 if not QUICK else 8]
print('high-cardinality cats', [(c, card[c]) for c in cat_use[:5]])

med = Xtr[num_cols].median(numeric_only=True)
Xtr_n = Xtr[num_cols].fillna(med).copy()
Xte_n = Xte[num_cols].fillna(med).copy()
for c in cat_use:
    freq = Xtr[c].astype(str).value_counts(normalize=True)
    Xtr_n[c + '_freq'] = Xtr[c].astype(str).map(freq).fillna(0.0)
    Xte_n[c + '_freq'] = Xte[c].astype(str).map(freq).fillna(0.0)

engineer = FE.AdvancedFeatureEngineer()
if cat_use:
    demo = Xtr[[cat_use[0]]].copy()
    demo = engineer.frequency_encode(demo, cat_use[0])
    print('FE.frequency_encode demo col', demo.columns[-1])
print('design matrix', Xtr_n.shape)


## 5. 单模型 QuickBaseline（log 空间）


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import mean_squared_error, mean_absolute_error

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_n, ytr, Xte_n, yte, task_type='regression')
comparison = baseline.compare_models(results, top_n=6)
display(comparison)
best_name = comparison.iloc[0]['Model']
pred_best = np.asarray(results[best_name]['predictions'])
print('best', best_name, 'RMSE(log)', mean_squared_error(yte, pred_best, squared=False))


## 6. `ModelEnsemble.blending`：树 + 线性

Blending 在训练集拟合基模型，在验证折上融合。  
回归任务下 `meta_model=None` 时返回**加权平均**预测函数（见 `wanglaoshi.CompetitionTools.ModelEnsemble`）。


In [ ]:
from wanglaoshi.CompetitionTools import ModelEnsemble
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge

X_fit, X_val, y_fit, y_val = train_test_split(
    Xtr_n, ytr, test_size=0.2, random_state=RANDOM_STATE + 1
)
models = [
    GradientBoostingRegressor(n_estimators=60 if QUICK else 120, random_state=RANDOM_STATE),
    RandomForestRegressor(n_estimators=80 if QUICK else 150, random_state=RANDOM_STATE, n_jobs=-1),
    Ridge(alpha=3.0),
]
ensemble = ModelEnsemble()
blend_pack = ensemble.blending(
    models=models,
    X_train=X_fit,
    y_train=y_fit,
    X_val=X_val,
    y_val=y_val,
    meta_model=None,
    weights=[0.45, 0.45, 0.10],
)
blend_pred = blend_pack['predict'](Xte_n)
single_rmse = mean_squared_error(yte, pred_best, squared=False)
blend_rmse = mean_squared_error(yte, blend_pred, squared=False)
print('holdout RMSE(log) single', round(single_rmse, 4), 'blend', round(blend_rmse, 4))
print('holdout MAE(price)', mean_absolute_error(np.expm1(yte), np.expm1(blend_pred)))


## 6.1 缺失指示列（Garage 语义）

若多列 `Garage*` 同时缺失，可新增 `HasGarage=0`；有值时 `HasGarage=1`。  
树模型可自动分裂，但显式指示列往往利于线性模型与可解释性。


In [ ]:
if garage_cols:
    miss_all = train[garage_cols].isna().all(axis=1).astype(int)
    print('rows all-Garage-missing', miss_all.sum(), '/', len(train))
    Xtr_n['HasGarage_hint'] = (~Xtr[garage_cols[:3]].isna().all(axis=1)).astype(int)
    Xte_n['HasGarage_hint'] = (~Xte[garage_cols[:3]].isna().all(axis=1)).astype(int)


## 6.2 集成理论：何时 Blending 有效

- **多样性**：GBDT 抓非线性，Ridge 抓全局线性趋势
- **过拟合风险**：Blending 权重在 hold-out 上调；正式比赛应使用 stacking + OOF
- `ModelEnsemble.stacking` 需要 `cv` 折 OOF 预测，比本章 Blending 更严但更稳


In [ ]:
print('单模型 top3:', comparison['Model'].head(3).tolist())
print('集成输出维度', np.asarray(blend_pred).shape)


## 7. 模拟提交（还原到价格空间）


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

price_pred = np.expm1(blend_pred)
price_true = np.expm1(yte.to_numpy())
out = Path('outputs/11_house_prices')
out.mkdir(parents=True, exist_ok=True)
ids = np.arange(1, len(price_pred) + 1)
sample = pd.DataFrame({'Id': ids, 'SalePrice': price_true})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    price_pred,
    str(out / 'sample_submission.csv'),
    str(out / 'submission.csv'),
    prediction_type='label',
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
sub.head(3)


## 8. 检查清单
- [x] 缺失含义意识（Garage 等）
- [x] log1p 目标与 expm1 提交
- [x] 高基数类别频率编码 + `FE.frequency_encode`
- [x] `ModelEnsemble.blending`
- [ ] 缺失指示列
- [ ] stacking 出折 OOF 预测再 meta


## 附录：房价回归方法论笔记

**log1p 与 RMSLE**：Kaggle House Prices 官方 metric 与 log 误差同源。训练在 log 空间，汇报时可同时给 `expm1` 后的 MAE 便于业务理解。

**高基数 Neighborhood**：频率编码是极简 baseline；更强方案是 TE（仍需 OOF）或 CatBoost。我们看到 `FE.frequency_encode` 与手写 `value_counts(normalize=True)` 等价，强调 **map 统计量来自 Xtr**。

**缺失结构**：`None` 在部分列表示「无该项设施」，不是 MCAR。缺失指示 + 填充「Unknown」水平，常比单纯 `median` 更好。

**ModelEnsemble**：`blending` 用固定验证子集；`stacking` 用 CV OOF，方差更小。权重 `[0.45,0.45,0.10]` 仅为示例，应在验证集上网格搜索。

**提交规范**：`SubmissionGenerator` 的 `prediction_type='label'` 输出美元价格点估计；勿把置信区间或 log 值直接写入 SalePrice 列。

**QUICK 模式**：子样本 + 浅树，RMSE 仅作流程验证；真要冲榜需全量特征工程 + 五折 stacking + 模型融合。


## 附录 B：数值特征与房价相关（抽样）


In [ ]:
corr_cols = [c for c in num_cols if c in Xtr.columns][:8]
if corr_cols:
    corrs = pd.concat([Xtr[corr_cols], ytr.rename('log_price')], axis=1).corr()['log_price'].drop('log_price')
    display(corrs.sort_values(ascending=False))


## 附录 C：`ModelEnsemble.stacking` 与本案关系


In [ ]:
# stacking 需要多折 OOF，QUICK 下仅打印 API 入口，避免长耗时
print('如需 stacking: ModelEnsemble().stacking(models, meta_model, X_train, y_train, cv=5)')
print('本案 blending 更适合课堂 5 分钟内跑通 evidence')


## 附录 D：SalePrice 分布与 log 变换动机


In [ ]:
import matplotlib.pyplot as plt
raw_y = np.expm1(ytr)
fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
ax[0].hist(raw_y, bins=30, color='steelblue', alpha=0.85)
ax[0].set_title('SalePrice 原尺度')
ax[1].hist(ytr, bins=30, color='coral', alpha=0.85)
ax[1].set_title('log1p(SalePrice)')
plt.tight_layout()
plt.show()
print('raw skew', pd.Series(raw_y).skew(), 'log skew', pd.Series(ytr).skew())


## 附录 E：高基数类别处理对照

| 方法 | 泄漏风险 | 适用模型 | 本案例 |
|------|----------|----------|--------|
| One-Hot Top-N | 低 | 线性/NN | 未用 |
| 频率编码 | 低（train map） | 树/线性 | ✅ |
| Target Encoding | 高（若全表） | 通用 | 见 Adult 案例 |
| CatBoost | 低（ordered） | 树 | 拓展 |

`ModelEnsemble.blending` 把多样性模型合成；当你仅有频率编码特征时，树模型往往单模最强，Ridge 提供线性基准防止过拟合。提交前 `expm1` 还原，并用 `SubmissionGenerator` 校验列名与 `Id` 对齐。


## 教师导读：House Prices 集成最小闭环

**关键词**：缺失语义、log 目标、频率编码、blending。

**实验板**：左侧单模 RF，右侧 `ModelEnsemble.blending`；对比 hold-out RMSE(log) 与 MAE(price)。集成未必必胜，但要学会**报告两种尺度**。

**高基数课**：打印 Neighborhood 基数；让学生想象 One-Hot 列数。频率编码是 leaky-safe 入门，TE 留到 E10。

**Garage 缺失**：白板画「无车库 ⇒ NA 合法」；HasGarage_hint 列是思想实验，不追求 leaderboard 增益。

**stacking vs blending**：用 2 分钟口径——blending 快、易过拟合权重；stacking 折 OOF 更贵更稳。QUICK 课堂选 blending 保障 evidence。

**提交 ritual**：`expm1` → `prediction_type='label'` → `validate_submission`。强调竞赛列是美元，不是 log。


## 附录 B：缺失的语义

`GarageYrBlt` 缺失常表示无车库，填 0 或中位数意义不同。应同时建 `GarageMissing` 指示列。

数值填补统计量：**仅训练集**。


In [ ]:
print('Checklist: indicator for structurally missing; median from train only; log1p target; CV before ensemble.')


## 附录 C：集成之前

Blending 需要独立验证折 `X_val`；Stacking 用 CV 造元特征。先让单模稳定，再 `ModelEnsemble`。


## 深度阅读与作业（House Prices）

### 必读观点
1. 对数目标 + RMSE(log) 是房价竞赛常规。
2. 结构性缺失要有指示列。
3. 集成前先有可复现单模；Blending 必须有独立验证折。

### 书面作业
1. 比较原始目标 vs log 目标的残差形态（文字+简图）。
2. 说明某缺失列的业务含义与填充策略。
3. 描述你调用 `ModelEnsemble` / `SubmissionGenerator` 的参数契约。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：11_house_prices

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。
